In [10]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr

from calm_data_generator.generators.clinical.Clinic import ClinicalDataGenerator
from calm_data_generator.tutorials.clinical_generator import build_correlation_matrix

SEED = 45
np.random.seed(SEED)

N_SAMPLES = 2000
print("calm_data_generator scenario -- seed:", SEED, "| n_samples:", N_SAMPLES)

calm_data_generator scenario -- seed: 45 | n_samples: 2000


In [11]:
group_names = ["A", "B", "D", "NS"]
group_sizes = [100, 200, 300, 400]
gene_groups = dict(zip(group_names, group_sizes))

n_demo = 4
custom_demo = {
    "INR": {"distribution": "truncnorm", "a": 0, "b": 5, "loc": 1.1, "scale": 0.3},
    "batch": {"distribution": "randint", "low": 1, "high": 5},  # 1,2,3,4
}

# Y excluded from correlation matrix -- Y se construye aparte via target_variable_config.
correlations_config = [
    {"internal": (0.3, 0.6), "demo_idx": 0, "demo_corr": 0.4},   # A & A <-> age
    {"internal": (0.2, 0.4), "demo_idx": 1, "demo_corr": 0.4},   # B & B <-> sex
    {"internal": (0.2, 0.3)},                                    # D
    {"internal": 0.0},                                           # NS
]

gene_groups

{'A': 100, 'B': 200, 'D': 300, 'NS': 400}

## 2. Matriz de correlación

In [12]:
corr_mx = build_correlation_matrix(n_demo, group_sizes, correlations_config)
print("corr_mx shape:", corr_mx.shape)  # (n_demo + n_genes, n_demo + n_genes)

corr_mx shape: (1004, 1004)


## 3. Generación

In [13]:
gen = ClinicalDataGenerator(seed=SEED)

datasets = gen.generate(
    n_samples=N_SAMPLES,
    n_genes=sum(group_sizes),
    gene_groups=gene_groups,
    custom_demographic_columns=custom_demo,
    demographic_gene_correlations=corr_mx,
    target_variable_config={
        "weights": {
            "Age":         3,
            "Sex_Binario": 1,
            "A":           2,
            "B":           5,
        },
        "binary_threshold": "median",
    },
)

demo_df = datasets["demographics"].copy()
genes_df = datasets["genes"].copy()

print("demographics columns:", list(demo_df.columns))
print("demographics shape  :", demo_df.shape)
print("genes shape          :", genes_df.shape)
print("gene_groups on gen   :", {k: (v[0], v[-1], len(v)) for k, v in gen.gene_groups.items()})
demo_df.head()

demographics columns: ['Age', 'Sex', 'INR', 'batch', 'Y']
demographics shape  : (2000, 5)
genes shape          : (2000, 1000)
gene_groups on gen   : {'A': ('G_0', 'G_99', 100), 'B': ('G_100', 'G_299', 200), 'D': ('G_300', 'G_599', 300), 'NS': ('G_600', 'G_999', 400)}


,Age,Sex,INR,batch,Y
Patient_ID,,,,,
PAT_79555_0,60,Male,1.265603,1.0,1
PAT_23544_1,29,Male,1.382265,4.0,0
PAT_32557_2,60,Male,1.475001,1.0,1
PAT_41129_3,54,Female,1.280068,1.0,0
PAT_83251_4,53,Female,1.369107,1.0,0


## 4. Correlaciones


In [14]:
Y = demo_df["Y"].astype(float).values

print("==== Group mean |correlation| of genes with Y ====")
print("Target: A~0.2  B~0.5  D~0  NS~0\n")

group_results = {}
for name in group_names:
    cols = gen.gene_groups[name]
    corrs = np.array([abs(pearsonr(genes_df[c].values, Y)[0]) for c in cols])
    group_results[name] = corrs
    print(f"  {name:<4} avg|r|={corrs.mean():.4f}  min|r|={corrs.min():.4f}  max|r|={corrs.max():.4f}  (n={len(cols)} genes)")

==== Group mean |correlation| of genes with Y ====
Target: A~0.2  B~0.5  D~0  NS~0

  A    avg|r|=0.2715  min|r|=0.2167  max|r|=0.3194  (n=100 genes)
  B    avg|r|=0.3167  min|r|=0.2670  max|r|=0.3997  (n=200 genes)
  D    avg|r|=0.0167  min|r|=0.0001  max|r|=0.0615  (n=300 genes)
  NS   avg|r|=0.0181  min|r|=0.0000  max|r|=0.0773  (n=400 genes)


In [15]:
print("==== Covariate correlation with Y ====")
print("Target: age~0.3  sex~0.1\n")

age = demo_df["Age"].astype(float).values
sex_bin = (demo_df["Sex"] == "Male").astype(int).values

r_age, _ = pearsonr(age, Y)
r_sex, _ = pearsonr(sex_bin, Y)
print(f"  age   r={r_age:.4f}")
print(f"  sex   r={r_sex:.4f}")

==== Covariate correlation with Y ====
Target: age~0.3  sex~0.1

  age   r=0.4370
  sex   r=0.4635


In [16]:
print("==== Sanity: intra-group / group<->demographic (mecanismo sin cambios) ====")
print("Target: age vs A ~0.40 | sex vs B ~0.40\n")

a_cols, b_cols = gen.gene_groups["A"], gen.gene_groups["B"]
a_age_r = np.mean([abs(pearsonr(genes_df[c].values, age)[0]) for c in a_cols])
b_sex_r = np.mean([abs(pearsonr(genes_df[c].values, sex_bin)[0]) for c in b_cols])
print(f"  age vs A(signal)  avg|r|={a_age_r:.4f}")
print(f"  sex vs B(signal)  avg|r|={b_sex_r:.4f}")

==== Sanity: intra-group / group<->demographic (mecanismo sin cambios) ====
Target: age vs A ~0.40 | sex vs B ~0.40

  age vs A(signal)  avg|r|=0.3686
  sex vs B(signal)  avg|r|=0.2956
